In [ ]:
# convnext_tiny - transfer learning
# Same flow as 02_cnn_baseline: grid search -> save winning config -> final training
# with early stopping -> evaluate on val -> evaluate on test ONCE -> save results.
#
# The heaviest of the four (~28M params). Noticeably slower on CPU -
# run this one last, and consider leaving it running while you do something else.

import sys
sys.path.insert(0, '..')

import os
import yaml
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from src.models.transfer_model import build_transfer_model
from src.data.split_dataset import check_dataset
from src.utils.metrics import get_predictions, show_confusion_matrix, show_classification_report

MODEL_NAME = "convnext_tiny"
CONFIG_PATH = "../configs/" + MODEL_NAME + "_config.yaml"
RESULTS_DIR = "../results/" + MODEL_NAME

os.makedirs(RESULTS_DIR, exist_ok=True)
print("Model:", MODEL_NAME)

In [ ]:
# check device - use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# make sure all three folders exist before going further
train_dir = "../data/processed/train"
val_dir = "../data/processed/val"
test_dir = "../data/processed/test"

for folder in [train_dir, val_dir, test_dir]:
    if not os.path.isdir(folder):
        print(folder, "not found, run 01_data_prep.ipynb first")

In [ ]:
# 224 (not 128 like the CNN) because these models were pretrained on 224x224 ImageNet images.
# The Normalize step uses ImageNet's mean/std - pretrained weights expect inputs
# scaled this way, so leaving it out costs real accuracy.
image_size = 224

transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_data = datasets.ImageFolder(train_dir, transform=transform)
val_data = datasets.ImageFolder(val_dir, transform=transform)
test_data = datasets.ImageFolder(test_dir, transform=transform)

num_classes = len(train_data.classes)
class_names = train_data.classes

print("Classes:", train_data.class_to_idx)
print("Train size:", len(train_data), "Val size:", len(val_data), "Test size:", len(test_data))

In [ ]:
# quick class balance check on train set - should be roughly equal after augmentation
counts = check_dataset(train_dir)

In [ ]:
# Grid search.
# Two differences from the CNN's search, both because this model starts from
# pretrained weights instead of from nothing (see RULEBOOK section 6):
#   - smaller learning rates: 0.001 / 0.0001 instead of 0.01 / 0.001
#   - freeze_backbone is searched instead of batch_size. Frozen = only the new
#     final layer trains (fast, less overfitting on a small dataset).
#     Unfrozen = the whole network adapts (slower, sometimes better).
# batch_size is fixed at 16 to keep this affordable on a CPU.

learning_rates = [0.001, 0.0001]
freeze_options = [True, False]
batch_size_search = 16
search_epochs = 3

best_val_acc = 0
best_settings = None

for lr in learning_rates:
    for freeze in freeze_options:
        print("Trying lr:", lr, "freeze_backbone:", freeze)

        # num_workers=0 matters on Windows - anything else can hang in a notebook
        train_loader = DataLoader(train_data, batch_size=batch_size_search, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_data, batch_size=batch_size_search, shuffle=False, num_workers=0)

        # fresh model each trial - don't reuse weights across trials
        model = build_transfer_model(MODEL_NAME, num_classes, freeze_backbone=freeze).to(device)
        loss_function = nn.CrossEntropyLoss()
        # only hand the optimizer the layers that are actually trainable
        trainable = [p for p in model.parameters() if p.requires_grad]
        optimizer = optim.Adam(trainable, lr=lr)

        for epoch in range(search_epochs):
            model.train()
            for images, labels in train_loader:
                images = images.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = loss_function(outputs, labels)
                loss.backward()
                optimizer.step()

        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = model(images)
                predicted = outputs.argmax(dim=1)
                val_correct += (predicted == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        print("  val acc:", round(val_acc, 3))

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_settings = {"learning_rate": lr, "freeze_backbone": freeze}

print()
print("Best settings:", best_settings, "with val acc:", round(best_val_acc, 3))

In [ ]:
# save the winning settings to this model's config file
# same idea as the CNN: the config file always matches what actually won the search

config = {
    "model_name": MODEL_NAME,
    "image_size": image_size,
    "batch_size": batch_size_search,
    "epochs": 30,
    "learning_rate": best_settings["learning_rate"],
    "freeze_backbone": best_settings["freeze_backbone"]
}

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config, f)

print("Saved to", CONFIG_PATH)
print(config)

In [ ]:
# load settings back from the config file
# from here on the config file is the single source of truth - nothing is hardcoded

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

batch_size = config['batch_size']
epochs = config['epochs']
learning_rate = config['learning_rate']
freeze_backbone = config['freeze_backbone']

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=0)

print("Training with:", config)

In [ ]:
# build the final model - fresh pretrained weights, not reused from the grid search
model = build_transfer_model(MODEL_NAME, num_classes, freeze_backbone=freeze_backbone).to(device)
loss_function = nn.CrossEntropyLoss()
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(trainable, lr=learning_rate)

total_params = sum(p.numel() for p in model.parameters())
print("Total parameters:", total_params)
print("Trainable parameters:", sum(p.numel() for p in trainable))

In [ ]:
# Final training loop.
# Early stopping here watches for val accuracy IMPROVING, and saves a checkpoint
# every time it hits a new best. That checkpoint is what gets evaluated below -
# the model sitting in memory at the end is just the last epoch, which is often
# worse than the best one.

model_path = os.path.join(RESULTS_DIR, "model.pt")

patience = 5
epochs_without_improvement = 0
best_val_acc = 0

for epoch in range(epochs):

    # training
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total

    # validation
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            predicted = outputs.argmax(dim=1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total

    print("Epoch", epoch + 1, "- train loss:", round(train_loss, 3),
          "train acc:", round(train_acc, 3), "val acc:", round(val_acc, 3))

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_without_improvement = 0
        torch.save(model.state_dict(), model_path)
        print("  new best val acc, model saved")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("Training stopped early - no val improvement for", patience, "epochs")
        break

print()
print("Best val accuracy during training:", round(best_val_acc, 3))

In [ ]:
# reload the best saved weights before evaluating
model.load_state_dict(torch.load(model_path))
print("Loaded best saved weights from", model_path)

In [ ]:
# check performance on val set - this is fine to look at during development
true_labels, predicted_labels = get_predictions(model, val_loader, device)

print("VAL SET RESULTS")
show_confusion_matrix(true_labels, predicted_labels, class_names)
show_classification_report(true_labels, predicted_labels, class_names)

In [ ]:
# TEST SET - the final, honest, one-time score.
# Only run this once you are fully done tuning. Do not go back and tune after
# seeing it (RULEBOOK section 6, "test set discipline").

test_true_labels, test_predicted_labels = get_predictions(model, test_loader, device)

print("TEST SET RESULTS (final)")
show_confusion_matrix(test_true_labels, test_predicted_labels, class_names)
show_classification_report(test_true_labels, test_predicted_labels, class_names)

In [ ]:
# save the test results - small text file, goes in git, this is the evidence trail
from sklearn.metrics import classification_report

report_text = classification_report(test_true_labels, test_predicted_labels, target_names=class_names)

with open(os.path.join(RESULTS_DIR, "metrics.txt"), "w") as f:
    f.write("Settings used: " + str(config) + "\n\n")
    f.write("Test set results:\n")
    f.write(report_text)

print("Saved to", os.path.join(RESULTS_DIR, "metrics.txt"))

In [ ]:
# model.pt is already saved in RESULTS_DIR by the training loop above.
# It is too large for git - upload it to the shared Drive models folder and add
# the link to results/README.md (see RULEBOOK section 4).
print("Trained weights are at:", model_path)